In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape

(898, 35)

In [5]:
X = df.drop("Class",axis=1)
y = df["Class"]

In [6]:
df["Class"].value_counts()

Class
DOKOL     204
SAFAVI    199
ROTANA    166
DEGLET     98
SOGAY      94
IRAQI      72
BERHI      65
Name: count, dtype: int64

In [7]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [8]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [9]:
le = LabelEncoder()
y = le.fit_transform(y)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)

## ANN

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [13]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [14]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [15]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## Model

In [16]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            #1st Hidden Layer :
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            #2nd Hidden Layer :
            nn.Linear(64, 64),
            nn.ReLU(),

            #3rd Hidden layer: : (Softmax is applied By Default)
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [17]:
model = ANN()

# loss & Optimizer

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the NN :

In [18]:
epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        # Clear the previous batch gradient accumalation
        optimizer.zero_grad()
        
        # Forward Propogation :
        outputs = model(xb)

        # Loss Calculation
        loss = criteria(outputs, yb)

        # Backward Propogation
        loss.backward()
        optimizer.step() # Params Update

        running_loss += loss.item()

    train_loss = running_loss/len(train_loader)
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.6371288351390674
epoch = 2/100, loss = 1.0739398106284763
epoch = 3/100, loss = 0.7334256185137707
epoch = 4/100, loss = 0.5433494319086489
epoch = 5/100, loss = 0.4386908722960431
epoch = 6/100, loss = 0.3852455978808196
epoch = 7/100, loss = 0.34751569900823676
epoch = 8/100, loss = 0.3083061226036238
epoch = 9/100, loss = 0.2896295034367105
epoch = 10/100, loss = 0.27076036645018536
epoch = 11/100, loss = 0.24817021968572037
epoch = 12/100, loss = 0.2466348260641098
epoch = 13/100, loss = 0.22317373331474222
epoch = 14/100, loss = 0.21538375641988672
epoch = 15/100, loss = 0.19945279087709344
epoch = 16/100, loss = 0.196165868445583
epoch = 17/100, loss = 0.1903734485740247
epoch = 18/100, loss = 0.17616889334243277
epoch = 19/100, loss = 0.17266775278941446
epoch = 20/100, loss = 0.1661580285948256
epoch = 21/100, loss = 0.16293527023947757
epoch = 22/100, loss = 0.15817834080561347
epoch = 23/100, loss = 0.1553355669197829
epoch = 24/100, loss = 0.150264511